# Chapter 7 — Ensemble Learning and Random Forests
Companion notebook to `part1_fundamentals/07_ensemble_random_forests.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (VotingClassifier, BaggingClassifier, RandomForestClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingRegressor)

In [ ]:
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

## Voting Classifier (soft voting, combining different algorithms)

In [ ]:
log_clf = LogisticRegression()
svm_clf = SVC(probability=True)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

voting_clf = VotingClassifier(
    estimators=[("lr", log_clf), ("rf", rf_clf), ("svc", svm_clf)], voting="soft")
voting_clf.fit(X_train, y_train)

for name, clf in [("LogReg", log_clf), ("RandomForest", rf_clf), ("SVC", svm_clf), ("Voting", voting_clf)]:
    clf.fit(X_train, y_train)
    print(f"{name:14s} test accuracy = {clf.score(X_test, y_test):.3f}")

## Single tree vs Bagging vs Random Forest

In [ ]:
tree_clf = DecisionTreeClassifier(random_state=42)
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                             max_samples=100, bootstrap=True, n_jobs=-1, random_state=42,
                             oob_score=True)
rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)

for name, clf in [("Single Tree", tree_clf), ("Bagging", bag_clf), ("Random Forest", rnd_clf)]:
    scores = cross_val_score(clf, X, y, cv=10)
    print(f"{name:14s} CV accuracy = {scores.mean():.3f} (+/- {scores.std():.3f})")

bag_clf.fit(X_train, y_train)
print("Bagging OOB score (free validation estimate):", bag_clf.oob_score_)

## Feature importance (free from tree ensembles)

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
rnd_clf_iris = RandomForestClassifier(n_estimators=500, random_state=42)
rnd_clf_iris.fit(iris["data"], iris["target"])
for name, score in sorted(zip(iris["feature_names"], rnd_clf_iris.feature_importances_),
                           key=lambda t: -t[1]):
    print(f"{name:20s} {score:.3f}")

## AdaBoost & Gradient Boosting with early stopping

In [ ]:
ada_clf = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200,
                              algorithm="SAMME.R", learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)
print("AdaBoost test accuracy:", ada_clf.score(X_test, y_test))

In [ ]:
np.random.seed(42)
X_gb = np.random.rand(200, 1) - 0.5
y_gb = 3 * X_gb[:, 0] ** 2 + 0.05 * np.random.randn(200)
X_gb_train, X_gb_val, y_gb_train, y_gb_val = train_test_split(X_gb, y_gb, random_state=42)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, learning_rate=0.1,
                                  random_state=42)
gbrt.fit(X_gb_train, y_gb_train)

errors = [np.mean((y_gb_val - y_pred) ** 2)
          for y_pred in gbrt.staged_predict(X_gb_val)]
best_n_estimators = np.argmin(errors) + 1
print("Best number of trees (early stopping via staged_predict):", best_n_estimators)

plt.plot(range(1, len(errors) + 1), errors)
plt.axvline(best_n_estimators, color="r", linestyle="--")
plt.xlabel("Number of trees"); plt.ylabel("Validation MSE")
plt.title("Finding the optimal n_estimators")
plt.show()

## ✍️ Practice
1. Compare single tree vs Bagging(500) vs RandomForest(500) CV accuracy above — how much does each step help?
2. Why does averaging correlated weak classifiers help less than averaging independent ones?
3. Debug: GradientBoostingRegressor(n_estimators=1000, lr=0.01) still improving at 1000 — what would you try?
4. Print rnd_clf_iris feature_importances_ (done above) — which feature matters least? Try dropping it.

In [ ]:
# Your code / notes here